In [13]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
from probeinterface import write_probeinterface, read_probeinterface

import os
import numpy as np

import warnings
warnings.filterwarnings('ignore')

import pandas as pd

In [14]:
files = sorted(os.listdir("/media/ubuntu/sda/data/mouse6/ns4/natural_image"))

In [15]:

recording_raw = se.read_blackrock(file_path=f'/media/ubuntu/sda/data/mouse6/ns4/natural_image/mouse6_021322_natural_image_001.ns4')
recording_recorded = recording_raw.remove_channels(["98", '31', '32'])


In [16]:
probe_30channel = read_probeinterface('/media/ubuntu/sda/data/probe.json')
recording_recorded = recording_recorded.set_probegroup(probe_30channel)

recording_cmr = recording_recorded
recording_f = spre.bandpass_filter(recording_recorded, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_f, freq=60)

print(recording_f)
recording_cmr = spre.common_reference(recording_f, reference="global", operator="median")
recording_cmr = recording_cmr.rename_channels(['A-000', 'A-001', 'A-002', 'A-003', 'A-004',
                               'A-005', 'A-006', 'A-007', 'A-008', 'A-009',
                               'A-0010', 'A-011', 'A-012', 'A-013', 'A-014',
                               'A-015', 'A-016', 'A-017', 'A-018', 'A-019',
                               'A-020', 'A-021', 'A-022', 'A-023', 'A-024',
                               'A-025', 'A-026', 'A-027', 'A-028', 'A-029'])

BandpassFilterRecording: 30 channels - 10000.0Hz - 1 segments - 40,000,100 samples 
                         4,000.01s (1.11 hours) - int16 dtype - 2.24 GiB


In [22]:
base_folder = '/media/ubuntu/sda/mouse_test/sorted/single_month_test'

for rep in range(1):
    output_folder = base_folder + f'/rep_{rep}_threshold_4'
    os.makedirs(output_folder, exist_ok=True)
    
    recording_preprocessed = recording_cmr.save(format="binary", n_jobs = 20)

    default_params = {
            'detect_sign': -1,  
            'adjacency_radius': 120, 
            'freq_min': 300,  
            'freq_max': 3000,
            'filter': True,
            'whiten': True,  
            'num_workers': 20,
            'clip_size': 50,
            'detect_threshold': 5,
            'detect_interval': 3,  
        }
    sorting_mountainsort = ss.run_sorter(sorter_name='mountainsort4',
                                    recording=recording_preprocessed,
                                    remove_existing_folder='True',
                                    folder=output_folder,
                                    **default_params)

    analyzer_mountainsort = si.create_sorting_analyzer(
        sorting=sorting_mountainsort, 
        recording=recording_preprocessed, 
        format='binary_folder', 
        folder=output_folder + '/analyzer_kilosort4_binary'
    )

    # 计算扩展信息
    extensions_to_compute = [
        "random_spikes",
        "waveforms",
        "noise_levels",
        "templates",
        "unit_locations",
        "spike_locations",
        "correlograms",
        "template_similarity"
    ]

    extension_params = {
        "unit_locations": {"method": "center_of_mass"},
        "spike_locations": {"ms_before": 0.1},
        "correlograms": {"bin_ms": 0.1},
        "template_similarity": {"method": "cosine_similarity"}
    }

    analyzer_mountainsort.compute(extensions_to_compute, extension_params=extension_params, n_jobs = 20)

    # 读取spikes.npy并检查无效的spike
    spikes_path = output_folder + "/analyzer_kilosort4_binary/sorting/spikes.npy"
    spikes = np.load(spikes_path)

    # 获取recording的总样本数
    total_samples = recording_f.get_num_samples()

    # 检查第一个和最后一个spike
    first_spike_valid = spikes[0]['sample_index'] >= 0
    last_spike_valid = spikes[-1]['sample_index'] < total_samples

    # 如果第一个或最后一个spike无效，删除所有无效的spike
    if not first_spike_valid or not last_spike_valid:
        # 创建有效spike的掩码：sample_index >= 0 且 < total_samples
        valid_mask = (spikes['sample_index'] >= 0) & (spikes['sample_index'] < total_samples)
        spikes_filtered = spikes[valid_mask]
        
        # 保存过滤后的spikes
        np.save(spikes_path, spikes_filtered)
        print(f"删除了 {len(spikes) - len(spikes_filtered)} 个无效的spike")
        print(f"原始spike数量: {len(spikes)}, 过滤后: {len(spikes_filtered)}")
    else:
        print("所有spike都在有效范围内")

    qm_params = sqm.get_default_qm_params()
    analyzer_mountainsort.compute("quality_metrics", qm_params, n_jobs = 20)

    # 导出到phy格式
    import spikeinterface.exporters as sexp
    sexp.export_to_phy(analyzer_mountainsort, output_folder + "/phy_folder_for_kilosort", verbose=True, n_jobs = 20)

Use cache_folder=/tmp/spikeinterface_cache/tmp9kr_durn/YINKBEXN
write_binary_recording 
engine=process - n_jobs=20 - samples_per_chunk=10,000 - chunk_memory=585.94 KiB - total_memory=11.44 MiB - chunk_duration=1.00s


write_binary_recording (workers: 20 processes): 100%|██████████| 4001/4001 [00:03<00:00, 1160.80it/s]


Mountainsort4 use the OLD spikeextractors mapped with NewToOldRecording


noise_level (workers: 20 processes): 100%|██████████| 20/20 [00:00<00:00, 584.98it/s]
Compute : spike_locations (workers: 20 processes): 100%|██████████| 4001/4001 [00:00<00:00, 8299.41it/s]


所有spike都在有效范围内


extract PCs (workers: 20 processes): 100%|██████████| 4001/4001 [00:17<00:00, 231.95it/s]


Run:
phy template-gui  /media/ubuntu/sda/mouse_test/sorted/single_month_test/rep_0_threshold_4/phy_folder_for_kilosort/params.py


In [23]:
import pickle

from spikeinterface.core import get_template_extremum_channel
import scipy.spatial.distance
from scipy.sparse.csgraph import connected_components

In [25]:
for rep in range(1):
    output_base = f'/media/ubuntu/sda/mouse_test/sorted/single_month_test/rep_{rep}_threshold_4'
    sampling_frequency = recording_cmr.get_sampling_frequency()

    print(f"\n读取phy结果（不排除noise cluster）...")
    sorting_curated_phy = se.read_phy(f'{output_base}/phy_folder_for_kilosort', exclude_cluster_groups=['noise'])  # 不设置exclude_cluster_groups
    print(f"读取到 {len(sorting_curated_phy.unit_ids)} 个units\n")
    # 创建analyzer
    print("创建analyzer并计算extensions...")
    analyzer_curated_phy = si.create_sorting_analyzer(
        sorting=sorting_curated_phy, 
        recording=recording_cmr,  # 使用common reference后的recording
        format='binary_folder',
        folder=output_base + '/analyzer_curated_temp',
        n_jobs=20, 
        verbose=False
    )

    extensions_to_compute = [
        "random_spikes",
        "waveforms",
        "templates",
        "unit_locations",
        "template_similarity"
    ]

    extension_params = {
        "unit_locations": {"method": "center_of_mass"},
        "template_similarity": {"method": "cosine_similarity"}
    }

    analyzer_curated_phy.compute(extensions_to_compute, extension_params=extension_params, n_jobs=20, verbose = False)
    print("完成extensions计算\n")

    # 获取neuron信息（整个recording）
    templates_ext = analyzer_curated_phy.get_extension("templates")
    templates_dense = templates_ext.data["average"]
    sparsity = analyzer_curated_phy.sparsity
    unit_locations_ext = analyzer_curated_phy.get_extension("unit_locations")
    unit_locations = unit_locations_ext.get_data()
    channel_locations = analyzer_curated_phy.get_channel_locations()

    # 处理merge逻辑
    if unit_locations.shape[1] >= 2:
        unit_distances = scipy.spatial.distance.cdist(
            unit_locations[:, :2], 
            unit_locations[:, :2], 
            metric="euclidean"
        )
    else:
        unit_distances = scipy.spatial.distance.cdist(
            unit_locations, 
            unit_locations, 
            metric="euclidean"
        )

    template_similarity_ext = analyzer_curated_phy.get_extension("template_similarity")
    template_similarity = template_similarity_ext.get_data()

    distance_threshold = 10.0
    similarity_threshold = 0.95
    num_units = len(analyzer_curated_phy.unit_ids)
    pair_mask = np.zeros((num_units, num_units), dtype=bool)

    for i in range(num_units):
        for j in range(i + 1, num_units):
            if unit_distances[i, j] < distance_threshold and template_similarity[i, j] > similarity_threshold:
                pair_mask[i, j] = True
                pair_mask[j, i] = True

    n_components, labels = connected_components(
        csgraph=pair_mask, 
        directed=False, 
        return_labels=True
    )

    merge_unit_groups = []
    unit_ids_list = analyzer_curated_phy.unit_ids
    for component_id in range(n_components):
        unit_indices = np.where(labels == component_id)[0]
        if len(unit_indices) > 1:
            group = [unit_ids_list[i] for i in unit_indices]
            merge_unit_groups.append(group)

    # 应用merge（如果有需要merge的units）
    if len(merge_unit_groups) > 0:
        print(f"发现 {len(merge_unit_groups)} 组需要merge的units，开始merge...")
        analyzer_merged = analyzer_curated_phy.merge_units(
            merge_unit_groups=merge_unit_groups,
            censor_ms=0.3,
            merging_mode="hard",
            new_id_strategy="append",
            format='binary_folder',
            folder=output_base + '/analyzer_merged',
            verbose=True,
            n_jobs=20
        )
        
        analyzer_merged.compute(extensions_to_compute, extension_params=extension_params, n_jobs=20, verbose = False)
        
        templates_ext_final = analyzer_merged.get_extension("templates")
        templates_dense_final = templates_ext_final.data["average"]
        sparsity_final = analyzer_merged.sparsity
        unit_locations_ext_final = analyzer_merged.get_extension("unit_locations")
        unit_locations_final = unit_locations_ext_final.get_data()
        channel_locations_final = analyzer_merged.get_channel_locations()
        sorting_final = analyzer_merged.sorting
        unit_ids_list_final = analyzer_merged.unit_ids
        
        # 生成position_waveforms
        position_waveforms_final = []
        for unit_id in unit_ids_list_final:
            unit_index = analyzer_merged.sorting.id_to_index(unit_id)
            template_dense_unit = templates_dense_final[unit_index, :, :]
            template_sparse_unit = sparsity_final.sparsify_waveforms(template_dense_unit[np.newaxis, :, :], unit_id)[0]
            sparse_channel_indices = sparsity_final.unit_id_to_channel_indices[unit_id]
            
            if len(sparse_channel_indices) == 0:
                position_waveform = np.zeros(templates_dense_final.shape[1], dtype=templates_dense_final.dtype)
                position_waveforms_final.append(position_waveform)
                continue
            
            sparse_channel_locations = channel_locations_final[sparse_channel_indices, :2]
            unit_location = unit_locations_final[unit_index, :2]
            
            distances = np.sqrt(np.sum((sparse_channel_locations - unit_location[np.newaxis, :])**2, axis=1))
            epsilon = 1e-10
            weights = 1.0 / (distances + epsilon)
            weights = weights / np.sum(weights)
            
            position_waveform = np.dot(template_sparse_unit, weights)
            position_waveforms_final.append(position_waveform)
        
        position_waveforms_final = np.array(position_waveforms_final)
        extremum_channels_final = get_template_extremum_channel(
            analyzer_merged, 
            peak_sign="neg",
            outputs="id"
        )
        
        channel_ids_list = list(analyzer_merged.recording.get_channel_ids())
    else:
        print("无需merge units\n")
        # 不需要merge，使用原始结果
        templates_ext_final = analyzer_curated_phy.get_extension("templates")
        templates_dense_final = templates_ext_final.data["average"]
        sparsity_final = analyzer_curated_phy.sparsity
        unit_locations_ext_final = analyzer_curated_phy.get_extension("unit_locations")
        unit_locations_final = unit_locations_ext_final.get_data()
        channel_locations_final = analyzer_curated_phy.get_channel_locations()
        sorting_final = analyzer_curated_phy.sorting
        unit_ids_list_final = unit_ids_list
        
        # 生成position_waveforms
        position_waveforms_final = []
        for unit_id in unit_ids_list_final:
            unit_index = analyzer_curated_phy.sorting.id_to_index(unit_id)
            template_dense_unit = templates_dense_final[unit_index, :, :]
            template_sparse_unit = sparsity_final.sparsify_waveforms(template_dense_unit[np.newaxis, :, :], unit_id)[0]
            sparse_channel_indices = sparsity_final.unit_id_to_channel_indices[unit_id]
            
            if len(sparse_channel_indices) == 0:
                position_waveform = np.zeros(templates_dense_final.shape[1], dtype=templates_dense_final.dtype)
                position_waveforms_final.append(position_waveform)
                continue
            
            sparse_channel_locations = channel_locations_final[sparse_channel_indices, :2]
            unit_location = unit_locations_final[unit_index, :2]
            
            distances = np.sqrt(np.sum((sparse_channel_locations - unit_location[np.newaxis, :])**2, axis=1))
            epsilon = 1e-10
            weights = 1.0 / (distances + epsilon)
            weights = weights / np.sum(weights)
            
            position_waveform = np.dot(template_sparse_unit, weights)
            position_waveforms_final.append(position_waveform)
        
        position_waveforms_final = np.array(position_waveforms_final)
        extremum_channels_final = get_template_extremum_channel(
            analyzer_curated_phy, 
            peak_sign="neg",
            outputs="id"
        )
        
        channel_ids_list = list(analyzer_curated_phy.recording.get_channel_ids())

    # 计算每个unit的channel_id（template中值不为0的通道）
    # recording的channel_ids已经是probe的contact_ids（已在读取probe时设置）
    print("计算每个unit的channel_id...")
    channel_ids_dict = {}  # {unit_id: [contact_id1, contact_id2, ...]}
    for idx, unit_id in enumerate(unit_ids_list_final):
        unit_index = sorting_final.id_to_index(unit_id)
        template_unit = templates_dense_final[unit_index, :, :]  # (n_samples, n_channels)
        
        # 找到template中值不为0的通道
        non_zero_channels = []
        for ch_idx in range(template_unit.shape[1]):  # 遍历channels（最后一个维度）
            if np.any(template_unit[:, ch_idx] != 0):  # 检查该通道在所有时间点的值
                # recording的channel_id已经是contact_id，直接使用
                contact_id = str(channel_ids_list[ch_idx])
                non_zero_channels.append(contact_id)
        
        channel_ids_dict[unit_id] = non_zero_channels

    print(f"完成channel_id计算，共处理{len(channel_ids_dict)}个units\n")

    # 计算channel_snr（每个unit的各个channel的SNR）
    print("计算channel_snr...")
    n_channels = recording_cmr.get_num_channels()

    # 计算noise_std（使用前10秒的数据）
    duration_samples = int(10 * sampling_frequency)  # 10秒
    max_samples = min(duration_samples, recording_cmr.get_num_samples())
    traces = recording_cmr.get_traces(start_frame=0, end_frame=max_samples)  # (n_timepoints, n_channels)

    noise_std_detect = np.median(np.abs(traces) / 0.6745, axis=0)  # (n_channels,)

    all_spike_times = []
    all_spike_unit_ids = []
    for unit_id in unit_ids_list_final:
        spike_train = sorting_final.get_unit_spike_train(unit_id)
        all_spike_times.extend(spike_train.tolist())
        all_spike_unit_ids.extend([unit_id] * len(spike_train))

    n_spikes_total = len(all_spike_times)
    n_spikes_sample = min(1000, n_spikes_total)
    if n_spikes_sample > 0:
        random_indices = np.random.choice(n_spikes_total, size=n_spikes_sample, replace=False)
        sampled_spike_times = [all_spike_times[i] for i in random_indices]
        sampled_spike_unit_ids = [all_spike_unit_ids[i] for i in random_indices]
    else:
        sampled_spike_times = []
        sampled_spike_unit_ids = []

    # 提取这些spike的waveform并计算每个channel的负值amplitude
    left_sample = 10
    right_sample = 20
    window_size = left_sample + right_sample

    channel_snr_dict = {} 

    for unit_id in unit_ids_list_final:
        channel_snr_dict[unit_id] = {}
        unit_spike_times = [st for st, uid in zip(sampled_spike_times, sampled_spike_unit_ids) if uid == unit_id]
        
        if len(unit_spike_times) == 0:
            unit_spike_times = sorting_final.get_unit_spike_train(unit_id).tolist()
            if len(unit_spike_times) > 1000:
                unit_spike_times = np.random.choice(unit_spike_times, size=1000, replace=False).tolist()
        
        unit_waveforms = []  # List of (n_channels, window_size)
        valid_spike_times = []
        
        for spike_time in unit_spike_times:
            start = spike_time - left_sample
            end = spike_time + right_sample

            # recording_cmr合并后只有一个segment，直接使用全局采样点索引
            # 确保索引在有效范围内
            if start < 0:
                start = 0
            if end > recording_cmr.get_num_samples():
                end = recording_cmr.get_num_samples()
            
            waveform = recording_cmr.get_traces(start_frame=start, end_frame=end)  # (n_timepoints, n_channels)
            unit_waveforms.append(waveform)
            valid_spike_times.append(spike_time)
        
        if len(unit_waveforms) == 0:
            continue
        
        unit_waveforms = np.array(unit_waveforms)  # (n_spikes, n_timepoints, n_channels)
        
        spike_time_values = unit_waveforms[:, left_sample, :]  # (n_spikes, n_channels) - 每个spike在spike_time时刻各个channel的值
        
        channel_amplitudes = np.mean(spike_time_values, axis=0)  # (n_channels,) - 每个channel的平均值（在spike_time时刻）
        channel_snr = np.abs(channel_amplitudes) / noise_std_detect  # (n_channels,)
        
        # 只保存 channel_ids_dict[unit_id] 中列出的通道的 SNR
        unit_channel_ids = channel_ids_dict.get(unit_id, [])  # 获取该unit的channel_id列表
        
        for ch_idx, snr_value in enumerate(channel_snr):
            channel_id = str(channel_ids_list[ch_idx])
            # 只保存 channel_ids_dict 中列出的通道
            if channel_id in unit_channel_ids:
                channel_snr_dict[unit_id][channel_id] = float(snr_value)

    print(f"完成channel_snr计算，共处理{len(channel_snr_dict)}个units\n")

    # 生成整体的neuron_inf（所有units）
    neuron_inf_all = {}
    for idx, unit_id in enumerate(unit_ids_list_final):
        neuron_inf_all[unit_id] = {
            'location_x': float(unit_locations_final[idx, 0]),
            'location_y': float(unit_locations_final[idx, 1]),
            'position_waveform': position_waveforms_final[idx],
            'extremum_channel': extremum_channels_final[unit_id],
            'channel_id': channel_ids_dict[unit_id],
            'channel_snr': channel_snr_dict.get(unit_id, {})  # 添加channel_snr字段
        }

    # 生成detect_array（所有spikes）
    print("生成detect_array...")
    spike_vector_final = sorting_final.to_spike_vector()
    detect_data_all = []

    for spike in spike_vector_final:
        unit_index = spike['unit_index']
        unit_id = sorting_final.unit_ids[unit_index]
        sample_index = spike['sample_index']  # 采样点索引（单个session，从0开始）
        
        extremum_channel = extremum_channels_final[unit_id]
        
        detect_data_all.append({
            'time': sample_index,
            'unit_id': unit_id,
            'extremum_channel': str(extremum_channel),
        })

    detect_array_df = pd.DataFrame(detect_data_all)
    print(f"完成detect_array生成，共{len(detect_array_df)}个spikes\n")

    # 保存结果
    print("保存结果...")
    with open(output_base + '/neuron_inf.pickle', 'wb') as f:
        pickle.dump(neuron_inf_all, f)
    detect_array_df.to_csv(output_base + '/detect_array.csv', index=False)
    print(f"已保存到: {output_base}/neuron_inf.pickle 和 {output_base}/detect_array.csv\n")

    


读取phy结果（不排除noise cluster）...
读取到 54 个units

创建analyzer并计算extensions...


compute_waveforms (workers: 20 processes): 100%|██████████| 4001/4001 [00:22<00:00, 175.51it/s]


完成extensions计算

发现 2 组需要merge的units，开始merge...
compute_waveforms 
engine=process - n_jobs=20 - samples_per_chunk=10,000 - chunk_memory=585.94 KiB - total_memory=11.44 MiB - chunk_duration=1.00s


compute_waveforms (workers: 20 processes): 100%|██████████| 4001/4001 [00:23<00:00, 171.18it/s]


计算每个unit的channel_id...
完成channel_id计算，共处理52个units

计算channel_snr...
完成channel_snr计算，共处理52个units

生成detect_array...
完成detect_array生成，共1178444个spikes

保存结果...
已保存到: /media/ubuntu/sda/mouse_test/sorted/single_month_test/rep_0_threshold_4/neuron_inf.pickle 和 /media/ubuntu/sda/mouse_test/sorted/single_month_test/rep_0_threshold_4/detect_array.csv



In [28]:
paths = [f"/media/ubuntu/sda/mouse_test/sorted/single_month_test/rep_{r}/detect_array.csv" for r in range(5)]
detect_dfs = []
for path in paths:
    df = pd.read_csv(path, usecols=["time", "extremum_channel"])
    df = df[df["time"] < 3000000]
    df["extremum_channel"] = df["extremum_channel"].astype(str)
    detect_dfs.append(df)


def build_channel_times(df):
    return {ch: np.sort(g["time"].to_numpy()) for ch, g in df.groupby("extremum_channel")}


def count_matches(df_cmp, ref_map, tol=1):
    total = 0
    for ch, g in df_cmp.groupby("extremum_channel"):
        if ch not in ref_map:
            continue
        ref_times = ref_map[ch]
        cmp_times = g["time"].to_numpy()
        idx = np.searchsorted(ref_times, cmp_times)
        left_idx = np.clip(idx - 1, 0, len(ref_times) - 1)
        right_idx = np.clip(idx, 0, len(ref_times) - 1)
        left_match = np.abs(cmp_times - ref_times[left_idx]) <= tol
        right_match = np.abs(cmp_times - ref_times[right_idx]) <= tol
        total += np.logical_or(left_match, right_match).sum()
    return int(total)

n = len(detect_dfs)
percent_matrix = np.zeros((n, n), dtype=float)
ref_maps = [build_channel_times(df) for df in detect_dfs]
for i, ref_map in enumerate(ref_maps):
    for j, df_cmp in enumerate(detect_dfs):
        denom = len(df_cmp)
        if denom == 0:
            percent_matrix[j, i] = np.nan
            continue
        match_count = count_matches(df_cmp, ref_map, tol=1)
        percent_matrix[j, i] = match_count / denom * 100

percent_df = pd.DataFrame(percent_matrix, index=[f"rep_{r}" for r in range(n)], columns=[f"ref_rep_{r}" for r in range(n)])
percent_df.round(2)



,ref_rep_0,ref_rep_1,ref_rep_2,ref_rep_3,ref_rep_4
rep_0,100.00,97.72,96.91,97.16,98.18
rep_1,87.00,100.00,94.26,95.69,93.38
rep_2,84.96,92.82,100.00,95.29,93.19
rep_3,87.22,96.48,97.57,100.00,93.98
rep_4,84.21,89.95,91.16,89.79,100.00


In [29]:
th_path = "/media/ubuntu/sda/mouse_test/sorted/single_month_test/rep_0_threshold_4.5/detect_array.csv"
threshold_df = pd.read_csv(th_path, usecols=["time", "extremum_channel"])
threshold_df = threshold_df[threshold_df["time"] < 3000000]
threshold_df["extremum_channel"] = threshold_df["extremum_channel"].astype(str)
threshold_map = build_channel_times(threshold_df)

contain_percent = {}
for idx, df in enumerate(detect_dfs):
    denom = len(df)
    if denom == 0:
        contain_percent[f"rep_{idx}"] = np.nan
        continue
    match_count = count_matches(df, threshold_map, tol=1)
    contain_percent[f"rep_{idx}"] = match_count / denom * 100

pd.Series(contain_percent).round(2)



rep_0    96.27
rep_1    95.03
rep_2    95.19
rep_3    97.80
rep_4    90.40
dtype: float64

In [34]:
gt_path = "/media/ubuntu/sda/mouse_test/sorted/recordings_30_channel_12_months_mouse6_natim_full/clique_0/mouse6_021322_natural_image_001/gt_detect_array.csv"
gt_df = pd.read_csv(gt_path, usecols=["time", "extremum_channel"])
gt_df = gt_df[gt_df["time"] < 3000000]
gt_df["extremum_channel"] = gt_df["extremum_channel"].astype(str)
gt_map = build_channel_times(gt_df)

contain_gt_percent = {}
all_reps = detect_dfs + [threshold_df]
rep_labels = [f"rep_{r}" for r in range(len(detect_dfs))] + ["rep_0_threshold_4"]

denom_gt = len(gt_df)
for label, df in zip(rep_labels, all_reps):
    if denom_gt == 0:
        contain_gt_percent[label] = np.nan
        continue
    match_count = count_matches(gt_df, build_channel_times(df), tol=1)
    contain_gt_percent[label] = match_count / denom_gt * 100

pd.Series(contain_gt_percent).round(2)



rep_0                82.87
rep_1                84.68
rep_2                87.41
rep_3                86.03
rep_4                87.99
rep_0_threshold_4    86.64
dtype: float64